# Module 3: Light-Matter Interaction & Cavity QED
## Notebook 3.3: The Jaynes-Cummings Hamiltonian & RWA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qutip/computational-quantum-optics/blob/main/module_3_light_matter_cavity_qed/03_jaynes_cummings_hamiltonian.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/qutip/computational-quantum-optics)
[![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-brightgreen?logo=python)](https://www.python.org/)

---

### Computational Quantum Optics Curriculum
* **Module 0:** Mathematical & Quantum Foundations
* **Module 1:** Single-Mode Quantum States of Light
* **Module 2:** Phase Space Quasiprobability Distributions
* **Module 3:** **Light-Matter Interaction & Cavity QED**
  * **3.1** Two-Level Atoms & Optical Bloch Equations
  * **3.2** Rabi Oscillations, Detuning & Pulse Control
  * **3.3 The Jaynes-Cummings Hamiltonian & RWA** $\leftarrow$ *(You are here)*
  * **3.4** The Quantum Rabi Model & Dressed States
  * **3.5** Quantum Collapse and Revival of Atomic Inversion
  * **3.6** Resonance Fluorescence & The Mollow Triplet
  * **3.7** Electromagnetically Induced Transparency (EIT) & Dark States

---

### Learning Objectives
1. **First-Principles Cavity QED:** Quantize the single-mode cavity electromagnetic field and derive the Jaynes-Cummings Hamiltonian under the Rotating Wave Approximation (RWA).
2. **Conservation Laws & Subspace Invariance:** Prove that the total excitation operator $\hat{N} = \hat{a}^\dagger\hat{a} + \hat{\sigma}_+\hat{\sigma}_-$ commutes with $\hat{H}_{\text{JC}}$, decomposing the infinite Hilbert space into decoupled $2 \times 2$ blocks.
3. **Vacuum Rabi Oscillations:** Analyze the spontaneous coherent exchange of energy between an excited atom and an empty cavity mode ($n=0$) at rate $2g$.
4. **The $\sqrt{n+1}$ Quantization Signature:** Prove and numerically verify that the multi-photon Rabi frequency scales as $\Omega_n = 2g\sqrt{n+1}$—the definitive hallmark of field quantization.
5. **Atom-Field Entanglement Generation:** Compute the von Neumann entropy of the reduced atomic state and demonstrate periodic generation of maximally entangled atom-cavity Bell states.


In [ ]:
# Environment Setup: Silent Colab detection & high-DPI publication styling
import os
import sys

# Configure headless Agg backend to ensure bulletproof plotting across all environments
if "MPLBACKEND" not in os.environ:
    os.environ["MPLBACKEND"] = "Agg"

import numpy as np
import scipy.linalg as la
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

# Configure publication matplotlib styling
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "lines.linewidth": 2.0,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--"
})

try:
    import qutip as qt
    HAS_QUTIP = True
    print("QuTiP successfully imported.")
except ImportError:
    HAS_QUTIP = False
    print("QuTiP not found. Using native NumPy/SciPy implementations.")

try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not found. Interactive exploration will run in static fallback mode.")


---

## 1. Physical Motivation: Beyond the Semiclassical Approximation

In Notebooks 3.1 and 3.2, the optical field was treated semiclassically as an unperturbed classical wave $\vec{E}_0 \cos(\omega_L t)$. However, when an atom is placed inside a high-$Q$ optical or microwave resonator (cavity), the mode volume $V$ is microscopic, and the electric field per single photon reaches extreme values:
$$\mathcal{E}_{\text{vac}} = \sqrt{\frac{\hbar \omega_c}{2 \epsilon_0 V}}$$

Under these conditions, a single atom interacts with individual quanta of light. In 1963, **Edwin Jaynes and Fred Cummings** formulated the fundamental solvable quantum model describing this fully quantized interaction.

### The Full Atom-Cavity Dipole Hamiltonian
The total Hamiltonian consists of the bare cavity energy, the bare atomic energy, and the electric dipole coupling $-\hat{\vec{d}} \cdot \hat{\vec{E}}$:
$$\hat{H} = \hbar \omega_c \hat{a}^\dagger \hat{a} + \frac{\hbar \omega_a}{2} \hat{\sigma}_z + \hbar g (\hat{a} + \hat{a}^\dagger)(\hat{\sigma}_+ + \hat{\sigma}_-)$$
where:
* $\hat{a}, \hat{a}^\dagger$ are the cavity mode annihilation and creation operators ($[\hat{a}, \hat{a}^\dagger] = 1$).
* $\omega_c$ is the cavity resonance frequency.
* $\omega_a$ is the atomic transition frequency, and $\hat{\sigma}_z = |e\rangle\langle e| - |g\rangle\langle g|$.
* $g = -\frac{\vec{d}_{eg} \cdot \vec{\epsilon}}{\hbar} \mathcal{E}_{\text{vac}}$ is the **vacuum single-photon coupling strength**.

### The Rotating Wave Approximation (RWA)
Expanding the interaction term yields four physical processes:
$$\hbar g (\hat{a} + \hat{a}^\dagger)(\hat{\sigma}_+ + \hat{\sigma}_-) = \hbar g \underbrace{(\hat{a}\hat{\sigma}_+ + \hat{a}^\dagger\hat{\sigma}_-)}_{\text{energy-conserving}} + \hbar g \underbrace{(\hat{a}^\dagger\hat{\sigma}_+ + \hat{a}\hat{\sigma}_-)}_{\text{counter-rotating}}$$
* $\hat{a}\hat{\sigma}_+$: Atom absorbs a photon and is excited ($|g, n+1\rangle \to |e, n\rangle$).
* $\hat{a}^\dagger\hat{\sigma}_-$: Atom emits a photon into the cavity mode ($|e, n\rangle \to |g, n+1\rangle$).
* $\hat{a}^\dagger\hat{\sigma}_+$: Atom is excited while simultaneously creating a photon ($|g, n\rangle \to |e, n+1\rangle$, energy non-conserving by $+2\hbar\omega$).
* $\hat{a}\hat{\sigma}_-$: Atom de-excites while annihilating a photon ($|e, n+1\rangle \to |g, n\rangle$, energy non-conserving by $-2\hbar\omega$).

When coupling is in the standard regime ($g \ll \omega_c, \omega_a$), the counter-rotating terms rotate at frequency $\approx \pm 2\omega_c$ and average to zero. Discarding them yields the famous **Jaynes-Cummings Hamiltonian**:
$$\hat{H}_{\text{JC}} = \hbar \omega_c \hat{a}^\dagger \hat{a} + \frac{\hbar \omega_a}{2} \hat{\sigma}_z + \hbar g \left( \hat{a}\hat{\sigma}_+ + \hat{a}^\dagger\hat{\sigma}_- \right)$$

---

## 2. Invariant Subspaces & Block Diagonalization

Define the **total excitation number operator**:
$$\hat{N} = \hat{a}^\dagger \hat{a} + \hat{\sigma}_+ \hat{\sigma}_- = \hat{n} + \frac{\hat{I} + \hat{\sigma}_z}{2}$$

One can directly verify the fundamental commutation relation:
$$[\hat{H}_{\text{JC}}, \hat{N}] = 0$$

Because $\hat{N}$ is a conserved quantity of motion, the infinite-dimensional Hilbert space splits into completely decoupled finite-dimensional invariant subspaces:
1. **$N = 0$ Subspace:** The absolute ground state $|g, 0\rangle$ with energy $E_{g,0} = -\hbar\omega_a/2$.
2. **$N = n + 1$ Subspaces ($n \ge 0$):** Spanned by the two basis states:
   $$\mathcal{H}_n = \text{span} \left\{ |e, n\rangle, |g, n+1\rangle \right\}$$

Within each subspace $\mathcal{H}_n$, the Hamiltonian reduces to a simple $2 \times 2$ matrix:
$$\hat{H}_n = \hbar \begin{pmatrix} \omega_c n + \frac{\omega_a}{2} & g\sqrt{n+1} \\ g\sqrt{n+1} & \omega_c (n+1) - \frac{\omega_a}{2} \end{pmatrix}$$

Setting detuning $\Delta = \omega_a - \omega_c$, and defining energy offset $E_0^{(n)} = \hbar\omega_c(n + 1/2)$:
$$\hat{H}_n = E_0^{(n)} \hat{I} + \frac{\hbar}{2} \begin{pmatrix} \Delta & 2g\sqrt{n+1} \\ 2g\sqrt{n+1} & -\Delta \end{pmatrix}$$


---

## 3. Resonant Dynamics & The $\sqrt{n+1}$ Quantization Law

On resonance ($\Delta = \omega_a - \omega_c = 0$), the off-diagonal coupling $g\sqrt{n+1}$ drives coherent oscillations between the states $|e, n\rangle$ and $|g, n+1\rangle$.

If the system starts in the state $|\psi(0)\rangle = |e, n\rangle$ (atom excited, $n$ photons in the cavity), the state vector at time $t$ is:
$$|\psi(t)\rangle = \cos\left( g\sqrt{n+1} \, t \right) |e, n\rangle - i \sin\left( g\sqrt{n+1} \, t \right) |g, n+1\rangle$$

The probability of finding the atom in the excited state is:
$$P_e(t) = \cos^2\left( g\sqrt{n+1} \, t \right) = \frac{1 + \cos\left( 2g\sqrt{n+1} \, t \right)}{2}$$

The Rabi flopping frequency is:
$$\Omega_n = 2g\sqrt{n+1}$$

### Two Profound Physical Insights:
1. **The Vacuum Rabi Frequency ($n = 0$):**
   Even when the cavity initially contains **zero photons** ($n=0$), an excited atom still undergoes periodic Rabi oscillations at rate:
   $$\Omega_0 = 2g$$
   The atom spontaneously emits a photon into the cavity mode and then coherently re-absorbs it! This reversible spontaneous emission is the signature of the **strong coupling regime** ($g \gg \kappa, \gamma$, where $\kappa$ is cavity decay and $\gamma$ is atomic spontaneous decay into non-cavity modes).
2. **Quantum Entanglement Generation:**
   At time $t_B = \frac{\pi}{4g\sqrt{n+1}}$, the state becomes:
   $$|\psi(t_B)\rangle = \frac{1}{\sqrt{2}} \left( |e, n\rangle - i |g, n+1\rangle \right)$$
   This is a **maximally entangled atom-cavity Bell state**! Tracing out the cavity mode yields the reduced atomic density matrix:
   $$\hat{\rho}_a = \frac{1}{2} |e\rangle\langle e| + \frac{1}{2} |g\rangle\langle g| = \frac{1}{2}\hat{I}$$
   The von Neumann entanglement entropy reaches its absolute maximum:
   $$S(\hat{\rho}_a) = -\mathrm{Tr}(\hat{\rho}_a \ln \hat{\rho}_a) = \ln 2 \approx 0.6931 \text{ nats} = 1.0 \text{ bit}$$


In [ ]:
# Implementation: Full Jaynes-Cummings Numerical & Analytical Subspace Solver

class JaynesCummingsSimulator:
    """
    Simulates the quantized Jaynes-Cummings Hamiltonian:
    H = hbar*omega_c*a^dagger a + hbar*omega_a/2 * sigma_z + hbar*g*(a*sigma_+ + a^dagger*sigma_-)
    """
    def __init__(self, omega_c=1.0, omega_a=1.0, g=0.05, n_max=15):
        self.wc = float(omega_c)
        self.wa = float(omega_a)
        self.g = float(g)
        self.n_max = int(n_max)
        self.delta = self.wa - self.wc
        
        # Basis convention: basis states are ordered as |e, n> followed by |g, n>
        # Dimension = 2 * (n_max + 1)
        self.dim = 2 * (self.n_max + 1)
        self._build_hamiltonian()
        
    def _idx(self, state, n):
        """Maps atomic state ('e' or 'g') and photon number n to matrix index."""
        if state == 'e':
            return n
        else:
            return (self.n_max + 1) + n
            
    def _build_hamiltonian(self):
        """Constructs the full Jaynes-Cummings Hamiltonian matrix in bare product basis."""
        H = np.zeros((self.dim, self.dim), dtype=complex)
        
        # Diagonal elements: H0 = hbar*omega_c*n + hbar*omega_a/2 * sigma_z
        for n in range(self.n_max + 1):
            # |e, n>
            idx_e = self._idx('e', n)
            H[idx_e, idx_e] = self.wc * n + 0.5 * self.wa
            
            # |g, n>
            idx_g = self._idx('g', n)
            H[idx_g, idx_g] = self.wc * n - 0.5 * self.wa
            
        # Off-diagonal interaction: hbar*g * (a*sigma_+ + a^dagger*sigma_-)
        # a*sigma_+ couples |g, n+1> -> |e, n> with strength g*sqrt(n+1)
        # a^dagger*sigma_- couples |e, n> -> |g, n+1> with strength g*sqrt(n+1)
        for n in range(self.n_max):
            idx_e = self._idx('e', n)
            idx_g_next = self._idx('g', n + 1)
            coupling = self.g * np.sqrt(n + 1)
            H[idx_e, idx_g_next] = coupling
            H[idx_g_next, idx_e] = coupling
            
        self.H = H
        
    def solve_subspace(self, n, t_arr):
        """
        Analytical solution for invariant 2x2 subspace spanned by {|e, n>, |g, n+1>}.
        Returns Pe(t), Pg(t), and entanglement entropy S(t) starting from |e, n>.
        """
        omega_n = np.sqrt(self.delta**2 + 4.0 * self.g**2 * (n + 1))
        # Amplitude for |e, n>
        # c_e(t) = cos(omega_n*t/2) - i*(delta/omega_n)*sin(omega_n*t/2)
        pe = np.cos(0.5 * omega_n * t_arr)**2 + (self.delta / omega_n)**2 * np.sin(0.5 * omega_n * t_arr)**2
        pg = (4.0 * self.g**2 * (n + 1) / omega_n**2) * np.sin(0.5 * omega_n * t_arr)**2
        
        # Reduced atomic density matrix eigenvalues: lambda_1 = pe, lambda_2 = pg
        # Von Neumann entropy: S = - sum(p * ln(p))
        entropy = np.zeros_like(t_arr)
        for i in range(len(t_arr)):
            p1, p2 = pe[i], pg[i]
            s = 0.0
            if p1 > 1e-14:
                s -= p1 * np.log(p1)
            if p2 > 1e-14:
                s -= p2 * np.log(p2)
            entropy[i] = s
            
        return pe, pg, entropy
        
    def evolve_full_state(self, psi0, t_arr):
        """Propagates state vector psi0 under the full Hamiltonian via diagonalization."""
        eigvals, eigvecs = la.eigh(self.H)
        c0 = eigvecs.T.conj() @ psi0
        
        # psi(t) = sum_k c0_k * exp(-i * E_k * t) * v_k
        psi_t = np.zeros((len(t_arr), self.dim), dtype=complex)
        for k in range(self.dim):
            psi_t += np.outer(np.exp(-1j * eigvals[k] * t_arr), c0[k] * eigvecs[:, k])
            
        return psi_t

# Verify numerical diagonalization against analytical 2x2 formula
g_test = 0.1
sim_test = JaynesCummingsSimulator(omega_c=1.0, omega_a=1.0, g=g_test, n_max=5)
t_test = np.linspace(0.0, 50.0, 500)
pe_ana, pg_ana, s_ana = sim_test.solve_subspace(n=0, t_arr=t_test)

# Full numerical state propagation starting from |e, 0>
psi_init = np.zeros(sim_test.dim, dtype=complex)
psi_init[sim_test._idx('e', 0)] = 1.0
psi_history = sim_test.evolve_full_state(psi_init, t_test)
pe_num = np.abs(psi_history[:, sim_test._idx('e', 0)])**2

max_diff = np.max(np.abs(pe_ana - pe_num))
print("--- Jaynes-Cummings Simulator Verification ---")
print(f"Vacuum Rabi Frequency (2g):        {2.0 * g_test:.4f}")
print(f"Subspace Analytical vs. Full Error: {max_diff:.3e}")
assert max_diff < 1e-12, "Jaynes-Cummings numerical consistency check failed!"
print("Verification Passed: Full truncated Hilbert space agrees with exact invariant subspace to machine precision!")


In [ ]:
# Publication-Quality 4-Panel Visualization: Vacuum Rabi Flopping, sqrt(n+1) Scaling, Entanglement & Dressed Ladder

fig = plt.figure(figsize=(16, 12))
g_val = 1.0  # Normalized coupling strength

# =============================================================================
# Panel 1: Vacuum Rabi Oscillations (n = 0: Reversible Spontaneous Emission)
# =============================================================================
ax1 = fig.add_subplot(2, 2, 1)

sim_res = JaynesCummingsSimulator(omega_c=10.0, omega_a=10.0, g=g_val, n_max=10)
t_vac = np.linspace(0.0, 4.0 * np.pi / (2.0 * g_val), 500)
pe_0, pg_0, s_0 = sim_res.solve_subspace(n=0, t_arr=t_vac)

ax1.plot(g_val * t_vac, pe_0, 'b-', lw=2.2, label=r"Excited State Population $P_e(t) = |\langle e, 0|\psi(t)\rangle|^2$")
ax1.plot(g_val * t_vac, pg_0, 'r--', lw=2.2, label=r"Cavity Photon Population $P_g(t) = |\langle g, 1|\psi(t)\rangle|^2$")
ax1.axhline(0.5, color='gray', linestyle=':', alpha=0.7)

ax1.set_xlabel(r"Normalized Interaction Time ($g t$)")
ax1.set_ylabel("State Probability")
ax1.set_title(r"Vacuum Rabi Oscillations ($n=0$, $\Omega_0 = 2g$)")
ax1.set_ylim(-0.02, 1.08)
ax1.legend(loc="upper right", frameon=True)

# =============================================================================
# Panel 2: sqrt(n+1) Scaling: Hallmark of Field Quantization
# =============================================================================
ax2 = fig.add_subplot(2, 2, 2)

n_values = [0, 1, 3, 8]
colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728']
t_scale = np.linspace(0.0, 8.0, 600)

for n, col in zip(n_values, colors):
    pe_n, _, _ = sim_res.solve_subspace(n=n, t_arr=t_scale)
    om_n = 2.0 * g_val * np.sqrt(n + 1)
    ax2.plot(g_val * t_scale, pe_n, color=col, label=rf"$n = {n}$ ($\Omega_n = 2g\sqrt{{{n+1}}} \approx {om_n:.2f}g$)")

ax2.set_xlabel(r"Interaction Time ($g t$)")
ax2.set_ylabel(r"Atomic Inversion $P_e(t)$")
ax2.set_title(r"Field Quantization: $\Omega_n = 2g\sqrt{n+1}$ Rabi Frequency Scaling")
ax2.set_ylim(-0.02, 1.08)
ax2.legend(loc="upper right", frameon=True)

# =============================================================================
# Panel 3: Atom-Cavity Quantum Entanglement Entropy
# =============================================================================
ax3 = fig.add_subplot(2, 2, 3)

ax3.plot(g_val * t_vac, s_0 / np.log(2), 'purple', lw=2.5, label=r"Entanglement Entropy $S(\hat{\rho}_a)$")
ax3.axhline(1.0, color='gold', linestyle='--', lw=1.8, label=r"Max Bell State Entanglement ($1.0\text{ bit} = \ln 2$)")
ax3.axhline(0.0, color='k', linestyle=':', alpha=0.5, label="Separable Product State ($S = 0$)")

# Mark Bell state times
for k in range(4):
    t_bell = (2 * k + 1) * np.pi / (4.0 * g_val)
    ax3.axvline(g_val * t_bell, color='red', linestyle=':', alpha=0.6)
    if k == 0:
        ax3.text(g_val * t_bell + 0.05, 0.45, rf"$gt = \pi/4$" "\nBell State", color='red', fontsize=9)

ax3.set_xlabel(r"Interaction Time ($g t$)")
ax3.set_ylabel(r"Entropy $S(\hat{\rho}_a)$ [Bits]")
ax3.set_title("Atom-Field Entanglement Generation & Collapse")
ax3.set_ylim(-0.05, 1.15)
ax3.legend(loc="lower right", frameon=True)

# =============================================================================
# Panel 4: Energy Level Diagram of the Jaynes-Cummings Ladder
# =============================================================================
ax4 = fig.add_subplot(2, 2, 4)

# Draw bare vs. dressed levels for n = 0, 1, 2
max_ladder_n = 3
ladder_g = 0.25

for n in range(max_ladder_n):
    y_bare = n + 0.5
    # Bare uncoupled level (degenerate at Delta = 0)
    ax4.plot([0.5, 1.5], [y_bare, y_bare], 'k--', lw=1.5)
    ax4.text(0.1, y_bare, rf"$|e, {n}\rangle, |g, {n+1}\rangle$", verticalalignment='center', fontsize=9)
    
    # Dressed split levels: y_bare +/- g*sqrt(n+1)
    split = ladder_g * np.sqrt(n + 1)
    ax4.plot([2.5, 3.5], [y_bare + split, y_bare + split], 'b-', lw=2.2)
    ax4.plot([2.5, 3.5], [y_bare - split, y_bare - split], 'r-', lw=2.2)
    
    ax4.text(3.6, y_bare + split, rf"$|+, {n}\rangle$", color='b', verticalalignment='center', fontsize=9)
    ax4.text(3.6, y_bare - split, rf"$|-, {n}\rangle$", color='r', verticalalignment='center', fontsize=9)
    
    # Doublet arrow annotation
    ax4.annotate('', xy=(3.0, y_bare + split), xytext=(3.0, y_bare - split),
                 arrowprops=dict(arrowstyle='<->', color='purple', lw=1.5))
    ax4.text(2.6, y_bare, rf"$2g\sqrt{{{n+1}}}$", color='purple', fontsize=9, horizontalalignment='right')

# Ground state |g, 0>
ax4.plot([0.5, 1.5], [-0.3, -0.3], 'k-', lw=2.0)
ax4.plot([2.5, 3.5], [-0.3, -0.3], 'k-', lw=2.0)
ax4.text(0.1, -0.3, r"$|g, 0\rangle$ (Ground)", verticalalignment='center', fontsize=9)
ax4.text(3.6, -0.3, r"$|g, 0\rangle$ (Inert)", verticalalignment='center', fontsize=9)

ax4.set_xlim(0.0, 4.8)
ax4.set_ylim(-0.6, 3.6)
ax4.set_xticks([1.0, 3.0])
ax4.set_xticklabels(["Bare States ($g=0$)", "Dressed States ($g>0$)"])
ax4.set_ylabel(r"Excitation Manifold Energy ($\hbar\omega_c$)")
ax4.set_title("Jaynes-Cummings Dressed State Energy Ladder")

plt.tight_layout()
plt.show()


> **Physical Insight: Why Does the $\sqrt{n+1}$ Rabi Frequency Scaling Prove Light Is Quantized?**
>
> In classical electrodynamics, an electric field with amplitude $\mathcal{E}$ exerts a continuous torque on the atomic dipole, producing a Rabi frequency $\Omega_{\text{classical}} = d \mathcal{E} / \hbar \propto \sqrt{I}$ proportional to the field amplitude.
>
> In the quantum theory, however:
> 1. In absolute vacuum ($n = 0$), the classical field vanishes ($\mathcal{E} = 0$), so a classical atom would remain stationary in $|e\rangle$ forever without spontaneous cavity emission. In quantum optics, vacuum fluctuations induce the **Vacuum Rabi oscillation** with non-zero frequency $\Omega_0 = 2g$.
> 2. For an $n$-photon Fock state, the interaction Hamiltonian contains the creation/annihilation operator $\hat{a}^\dagger |n\rangle = \sqrt{n+1}|n+1\rangle$, giving discrete, quantized frequencies:
>    $$\Omega_n = 2g\sqrt{n+1}$$
> In 1996, **Serge Haroche, Jean-Michel Raimond, and Michel Brune** (ENS Paris) sent Rydberg rubidium atoms one-by-one through a superconducting microwave cavity containing 0, 1, or 2 photons. They observed discrete peaks in the Fourier spectrum of atomic inversion at precisely $\Omega_0$, $\sqrt{2}\Omega_0$, and $\sqrt{3}\Omega_0$, providing one of the most celebrated and direct experimental proofs of field quantization in the history of physics!


In [ ]:
# Interactive Exploration: Jaynes-Cummings Dynamics with Detuning & Photon Number

def plot_interactive_jc(n_photons=0, delta_ratio=0.0):
    """
    Simulates the Jaynes-Cummings oscillation for arbitrary Fock state n
    and atomic detuning Delta = delta_ratio * g.
    """
    g_sim = 1.0
    delta_sim = delta_ratio * g_sim
    sim = JaynesCummingsSimulator(omega_c=10.0, omega_a=10.0 + delta_sim, g=g_sim, n_max=n_photons + 2)
    
    t_range = np.linspace(0.0, 15.0, 500)
    pe, pg, ent = sim.solve_subspace(n=n_photons, t_arr=t_range)
    
    omega_n = np.sqrt(delta_sim**2 + 4.0 * g_sim**2 * (n_photons + 1))
    p_max = 4.0 * g_sim**2 * (n_photons + 1) / omega_n**2
    
    fig, (ax_pop, ax_ent) = plt.subplots(1, 2, figsize=(14, 4.5))
    
    # Population dynamics
    ax_pop.plot(t_range, pe, 'b-', lw=2.2, label=r"$P_e(t) = |\langle e, n|\psi\rangle|^2$")
    ax_pop.plot(t_range, pg, 'r--', lw=2.2, label=r"$P_g(t) = |\langle g, n+1|\psi\rangle|^2$")
    ax_pop.axhline(1.0 - p_max, color='gray', linestyle=':', label=rf"Minimum Inversion = {1.0 - p_max:.2f}")
    ax_pop.set_xlabel(r"Interaction Time ($g t$)")
    ax_pop.set_ylabel("Probability")
    ax_pop.set_title(rf"JC Dynamics ($n={n_photons}$, $\Delta = {delta_ratio:.1f}g$, $\Omega_n = {omega_n:.2f}g$)")
    ax_pop.set_ylim(-0.05, 1.05)
    ax_pop.legend(frameon=True)
    
    # Entanglement entropy
    ax_ent.plot(t_range, ent / np.log(2), 'purple', lw=2.2, label=r"Entropy $S(\hat{\rho}_a)$ [Bits]")
    ax_ent.axhline(1.0, color='gold', linestyle='--', label="Maximal 1-bit Entanglement")
    ax_ent.set_xlabel(r"Interaction Time ($g t$)")
    ax_ent.set_ylabel("Von Neumann Entropy [Bits]")
    ax_ent.set_title("Atom-Field Entanglement Oscillations")
    ax_ent.set_ylim(-0.05, 1.05)
    ax_ent.legend(frameon=True)
    
    plt.tight_layout()
    plt.show()

if HAS_WIDGETS:
    widgets.interact(
        plot_interactive_jc,
        n_photons=widgets.IntSlider(value=0, min=0, max=5, step=1, description="Fock state $n$:"),
        delta_ratio=widgets.FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.5, description=r"Detuning $\Delta/g$:")
    )
else:
    print("Static exploration demonstration (default parameters: n = 1, Delta = 0.5 g):")
    plot_interactive_jc(n_photons=1, delta_ratio=0.5)


---

## 4. Self-Check Exercises & Guided Challenge

### Challenge 1: The Purity of the Reduced Atomic Density Matrix
At time $t_B = \pi / (4g)$ in the resonant vacuum Rabi oscillation ($n=0$, $\Delta = 0$), the composite system is in the state:
$$|\psi(t_B)\rangle = \frac{1}{\sqrt{2}} |e, 0\rangle - \frac{i}{\sqrt{2}} |g, 1\rangle$$

1. Calculate the reduced atomic density matrix $\hat{\rho}_a = \mathrm{Tr}_{\text{cavity}}(|\psi\rangle\langle\psi|)$.
2. Compute the state purity $\gamma = \mathrm{Tr}(\hat{\rho}_a^2)$ and verify that it equals $0.5$ (the minimum possible purity for a two-level system).

<details>
<summary><b>Solution & Derivation</b> (Click to expand)</summary>

The composite density matrix is:
$$\hat{\rho} = |\psi\rangle\langle\psi| = \frac{1}{2} |e, 0\rangle\langle e, 0| + \frac{1}{2} |g, 1\rangle\langle g, 1| - \frac{i}{2} |e, 0\rangle\langle g, 1| + \frac{i}{2} |g, 1\rangle\langle e, 0|$$
Tracing over the cavity Fock basis $\{|0\rangle, |1\rangle\}$:
$$\hat{\rho}_a = \langle 0|\hat{\rho}|0\rangle + \langle 1|\hat{\rho}|1\rangle = \frac{1}{2} |e\rangle\langle e| + \frac{1}{2} |g\rangle\langle g| = \begin{pmatrix} 1/2 & 0 \\ 0 & 1/2 \end{pmatrix} = \frac{1}{2}\hat{I}$$
The coherence terms vanish because $\langle 0|g, 1\rangle = \langle 1|e, 0\rangle = 0$.
The purity is:
$$\mathrm{Tr}(\hat{\rho}_a^2) = \left(\frac{1}{2}\right)^2 + \left(\frac{1}{2}\right)^2 = \frac{1}{4} + \frac{1}{4} = \frac{1}{2} = 0.5$$
The atom is in a **maximally mixed state**, which proves that it is **maximally entangled** with the cavity field!
</details>

---

### Challenge 2: Numerical Verification of Purity and Entropy
Simulate resonant vacuum Rabi flopping ($n=0$) and compute the exact numerical atomic purity $\mathrm{Tr}(\hat{\rho}_a^2)$ and von Neumann entropy at time $t_B = \pi / (4g)$. Verify that purity equals $0.500000$ and entropy equals $\ln 2 \approx 0.693147$ nats.


In [ ]:
# Numerical Verification of Challenge 2: Bell State Entanglement & Purity Benchmark

g_val = 1.0
t_bell = np.pi / (4.0 * g_val)

sim_bell = JaynesCummingsSimulator(omega_c=10.0, omega_a=10.0, g=g_val, n_max=3)
pe_b, pg_b, ent_b = sim_bell.solve_subspace(n=0, t_arr=np.array([t_bell]))

pe_val = pe_b[0]
pg_val = pg_b[0]
entropy_val = ent_b[0]

# Compute purity Tr(rho_a^2) = pe^2 + pg^2
purity_val = pe_val**2 + pg_val**2

print(f"--- Bell State Entanglement Benchmark (gt = pi/4 = {g_val*t_bell:.4f}) ---")
print(f"Excited State Population Pe: {pe_val:.6f} (Expected 0.500000)")
print(f"Ground State Population Pg:  {pg_val:.6f} (Expected 0.500000)")
print(f"Atomic State Purity:         {purity_val:.6f} (Expected 0.500000)")
print(f"Entanglement Entropy:        {entropy_val:.6f} nats (Expected ln(2) = {np.log(2):.6f})")

assert abs(pe_val - 0.5) < 1e-12, "Population balance check failed!"
assert abs(purity_val - 0.5) < 1e-12, "Purity 0.5 check failed!"
assert abs(entropy_val - np.log(2)) < 1e-12, "Entropy ln(2) check failed!"
print("Verification Passed: Atom is maximally entangled with cavity field to machine precision!")


---

## 5. Summary & Key Formulas

| Quantity | Mathematical Expression | Physical Significance |
| :--- | :--- | :--- |
| **JC Hamiltonian** | $\hat{H}_{\text{JC}} = \hbar\omega_c\hat{a}^\dagger\hat{a} + \frac{\hbar\omega_a}{2}\hat{\sigma}_z + \hbar g(\hat{a}\hat{\sigma}_+ + \hat{a}^\dagger\hat{\sigma}_-)$ | Quantized atom-cavity interaction under RWA |
| **Conserved Quantity** | $\hat{N} = \hat{a}^\dagger\hat{a} + \hat{\sigma}_+\hat{\sigma}_-$ ($[\hat{H}_{\text{JC}}, \hat{N}] = 0$) | Total excitation conservation; block diagonalizes $\mathcal{H}$ |
| **$n$-Photon Rabi Frequency** | $\Omega_n = 2g\sqrt{n+1}$ | Field quantization scaling law |
| **Vacuum Rabi Frequency** | $\Omega_0 = 2g$ | Spontaneous emission and reabsorption in vacuum |
| **Maximal Entanglement Time**| $t_B = \frac{\pi}{4g\sqrt{n+1}}$ | Generation of maximally entangled Bell state ($S = \ln 2$) |

---

### Further Reading & Foundational Literature
* **E. T. Jaynes & F. W. Cummings**, *Comparison of Quantum and Semiclassical Radiation Theories with Application to the Beam Maser*, Proc. IEEE 51, 89 (1963).
* **M. Brune, F. Schmidt-Kaler, A. Maali, J. Dreyer, E. Hagley, J. M. Raimond, & S. Haroche**, *Quantum Rabi Oscillation: A Direct Test of Field Quantization in a Cavity*, Phys. Rev. Lett. 76, 1800 (1996).
* **S. Haroche & J. M. Raimond**, *Exploring the Quantum: Atoms, Cavities, and Photons*, Oxford University Press (2006).

---

### Next Notebook in Module 3
In **Notebook 3.4: The Quantum Rabi Model & Dressed States**, we push beyond the Rotating Wave Approximation into the **ultra-strong coupling (USC)** and **deep strong coupling (DSC)** regimes ($g/\omega_c \gtrsim 0.1$):
* Restoring the counter-rotating terms $\hat{a}\hat{\sigma}_- + \hat{a}^\dagger\hat{\sigma}_+$: The full **Quantum Rabi Model (QRM)**.
* Breakdown of excitation number conservation ($[\hat{H}_{\text{QRM}}, \hat{N}] \ne 0$) and $\mathbb{Z}_2$ parity symmetry.
* Virtual ground-state photons and dressed eigenstate hybridization.
